In [ ]:
MPB:
Added session_status_id to the MPB block using rdmstatus.session_status_id from silver_rdm_session_status.

WIP:
Added session_status_id to the WIP block using rdmstatus.session_status_id from silver_rdm_session_status.

SONE / S1:
Added session_status_id to the SONE block using rdmstatus.session_status_id from silver_rdm_session_status.

In [ ]:
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN deleted_at IS NULL THEN 1 ELSE 0 END) AS null_count,
    SUM(CASE WHEN deleted_at IS NOT NULL THEN 1 ELSE 0 END) AS not_null_count
FROM silver_drj_appointments;

In [ ]:
CASE WHEN appt.deleted_at IS NOT NULL THEN 0 ELSE 1 END AS z_src_is_active

In [ ]:
SELECT
    z_src_is_active,
    COUNT(*) AS row_count
FROM silver_sessions
WHERE z_src_system_id = 'MPB'
GROUP BY z_src_is_active
ORDER BY z_src_is_active;

In [ ]:
Updated the MPB block in the Sessions notebook to add z_src_is_active as per Monday definition. Logic implemented as 0 when appt.deleted_at is present and 1 otherwise. Validated the output by checking the active/inactive record split after execution.

In [ ]:
SELECT
    serv.description,
    CASE
        WHEN LOWER(TRIM(serv.description)) IN (
            'case raised in error',
            'bnssg - case raised in error'
        ) THEN 0
        ELSE 1
    END AS z_src_is_active,
    COUNT(*) AS row_count
FROM silver_wip_activityentry ae
LEFT JOIN silver_wip_activityservice actserv
    ON actserv.id = ae.activity_service_id
LEFT JOIN silver_wip_service serv
    ON serv.id = actserv.service_id
WHERE LOWER(TRIM(serv.description)) IN (
    'case raised in error',
    'bnssg - case raised in error'
)
GROUP BY
    serv.description,
    CASE
        WHEN LOWER(TRIM(serv.description)) IN (
            'case raised in error',
            'bnssg - case raised in error'
        ) THEN 0
        ELSE 1
    END;

In [ ]:
SELECT DISTINCT
    description
FROM silver_wip_service
WHERE LOWER(TRIM(description)) IN (
    'case raised in error',
    'bnssg - case raised in error'
);

In [ ]:
,CASE
    WHEN LOWER(TRIM(serv.description)) IN (
        'case raised in error',
        'bnssg - case raised in error'
    ) THEN 0
    ELSE 1
 END AS z_src_is_active

In [ ]:
Updated the WIP block in the Sessions notebook to add z_src_is_active as per Monday definition. Implemented logic using WIP service description, where sessions linked to Case raised in error / BNSSG - Case Raised in Error are flagged as 0, and all other sessions are flagged as 1. Validated source value variants before implementation.

In [ ]:
SELECT
    z_src_is_active,
    COUNT(*) AS row_count
FROM silver_sessions
WHERE z_src_system_id = 'WIP'
GROUP BY z_src_is_active
ORDER BY z_src_is_active;

In [ ]:
nnw

In [ ]:
SELECT
    z_src_system_id,
    COUNT(*) AS row_count
FROM silver_sessions_ytest
GROUP BY z_src_system_id
ORDER BY z_src_system_id;

In [ ]:
SELECT COUNT(*) AS wip_row_count
FROM silver_sessions_ytest
WHERE z_src_system_id = 'WIP';

In [ ]:
SELECT
    z_src_is_active,
    COUNT(*) AS row_count
FROM silver_sessions_ytest
WHERE z_src_system_id = 'WIP'
GROUP BY z_src_is_active
ORDER BY z_src_is_active;

In [ ]:
The WIP logic uses the service description values available in the source table already referenced in the notebook join path. The required values, including Case raised in error and BNSSG - Case Raised in Error, were validated in silver_rdm_wip_service_type, so this table was used to implement z_src_is_active.

In [ ]:
-- Updated WIP cprod logic to match RDM care product source-id build
,CONCAT(CAST(ah.service_type_id AS STRING),'_',CAST(actserv.service_id AS STRING),'_',CAST(ae.activity_type_id AS STRING)) AS session_cprod_src_id
,cp.cprod_id AS session_cprod_id

In [ ]:
,cp.cprod_id AS session_cprod_id

In [ ]:
-- Join to RDM care product using WIP cprod source-id logic
LEFT JOIN silver_rdm_care_product cp ON cp.cprod_src_sys_inst_id = 'WIP001' AND LOWER(TRIM(cp.cprod_src_id)) = LOWER(TRIM(CONCAT(CAST(ah.service_type_id AS STRING),'_',CAST(actserv.service_id AS STRING),'_',CAST(ae.activity_type_id AS STRING))))

In [ ]:
SELECT COUNT(*) AS total_rows,
       SUM(CASE WHEN session_cprod_id IS NOT NULL THEN 1 ELSE 0 END) AS mapped_count,
       SUM(CASE WHEN session_cprod_id IS NULL THEN 1 ELSE 0 END) AS unmapped_count
FROM silver_sessions_ytest
WHERE z_src_system_id = 'WIP'

In [ ]:
WIP
Updated the WIP block in the Sessions notebook for session_cprod_id.
Added/updated the following logic in the SELECT section:
CONCAT(CAST(ah.service_type_id AS STRING),'_',CAST(actserv.service_id AS STRING),'_',CAST(ae.activity_type_id AS STRING)) AS session_cprod_src_id
cp.cprod_id AS session_cprod_id
Added the following join in the JOIN section:
LEFT JOIN silver_rdm_care_product cp ON cp.cprod_src_sys_inst_id = 'WIP001' AND LOWER(TRIM(cp.cprod_src_id)) = LOWER(TRIM(CONCAT(CAST(ah.service_type_id AS STRING),'_',CAST(actserv.service_id AS STRING),'_',CAST(ae.activity_type_id AS STRING))))
Implemented using the same source-id build logic as the RDM Care Product mapping logic for WIP.

In [ ]:
session_cprod_id was taken from silver_rdm_care_product, but because that final RDM ID does not directly exist in the source session tables, the matching source care product key had to be built first as session_cprod_src_id. That key was then matched to silver_rdm_care_product.cprod_src_id, and the mapped cprod_id was selected as session_cprod_id.


Built session_cprod_src_id in the session query, joined it to silver_rdm_care_product.cprod_src_id, and then pulled cp.cprod_id as session_cprod_id.

In [ ]:
SELECT
    cp.cprod_src_sys_inst_id,
    cp.cprod_src_id,
    COUNT(*) AS row_count
FROM silver_rdm_care_product cp
WHERE cp.cprod_src_sys_inst_id = 'WIP001'
GROUP BY cp.cprod_src_sys_inst_id, cp.cprod_src_id
HAVING COUNT(*) > 1

In [ ]:
SELECT COUNT(*) AS total_rows
FROM silver_sessions_ytest1
WHERE z_src_system_id = 'WIP'